# Task 3 - Domain Generalization

PACS ERM reuse, source-only alignment, and SAM without Sketch access before final evaluation.

**Execution policy:** this notebook is intentionally delivered unexecuted. Set the configuration paths and switches, then run top-to-bottom when you are ready to conduct the experiment. It does not answer the report questions.

## 1. Configuration and strict unseen-target protocol

No Sketch files are loaded until the explicitly marked final-evaluation section. This notebook reuses Task 2's fixed source split and Source-only checkpoint.

The next cell records source-only configurations and paths. Sketch is deliberately absent from the configuration and all subsequent training cells.

In [ ]:
# Standard library and experiment dependencies.
# Install dependencies yourself before executing: torch torchvision open_clip_torch
# scikit-learn pandas matplotlib seaborn pillow scipy tqdm
import json, random, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

SEED = 6304
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd().resolve()
if not (ROOT / "ATML-PA1.pdf").exists():
    ROOT = ROOT.parent


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)


def show_table(rows, title=None):
    frame = pd.DataFrame(rows)
    if title:
        print(title)
    display(frame)
    return frame


set_seed()

from torchvision.models import resnet18, ResNet18_Weights

CONFIG = {
    "pacs_root": ROOT / "data" / "PACS",
    "split_file": ROOT / "common" / "splits" / "pacs_sketch_seed6304.json",
    "erm_checkpoint": ROOT / "task2" / "results" / "source_only.pt",
    "results_dir": ROOT / "task3" / "results",
    "epochs": 30,
    "patience": 5,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "batch_per_domain": 8,
    "lambda_dg": 1.0,
    "rho": 0.05,
    "study_rhos": [0.01, 0.05, 0.1],
    "plot": True,
    "print_metrics": True,
}
SOURCES = ["photo", "art_painting", "cartoon"]
CLASSES = ["dog", "elephant", "giraffe", "guitar", "horse", "house", "person"]
RESULTS = Path(CONFIG["results_dir"])
RESULTS.mkdir(parents=True, exist_ok=True)


## 2. Source-only data utilities (Sketch deliberately absent)

The next cell loads only Photo, Art Painting, and Cartoon plus the saved Task 2 split. It defines preprocessing, the ResNet-18 feature path, MMD, and source-side evaluation.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
train_tf = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(weights.transforms().mean, weights.transforms().std),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(weights.transforms().mean, weights.transforms().std),
    ]
)


# Only the three observed source directories are enumerated in this function.
def source_index():
    records = []
    for d in SOURCES:
        for y, c in enumerate(CLASSES):
            records += [
                {"path": str(p), "domain": d, "label": y}
                for p in sorted((CONFIG["pacs_root"] / d / c).glob("*"))
            ]
    frame = pd.DataFrame(records)
    saved = pd.DataFrame(json.load(open(CONFIG["split_file"])))
    return frame.merge(saved[["path", "split"]], on="path", how="left", validate="one_to_one")


source = source_index()


class PACS(Dataset):
    def __init__(self, frame, tf):
        self.frame = frame.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        r = self.frame.iloc[i]
        return self.tf(Image.open(r.path).convert("RGB")), int(r.label), r.domain


def freeze_bn(m):
    for module in m.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def make_model():
    m = resnet18(weights=weights)
    m.fc = nn.Linear(512, 7)
    return m.to(DEVICE)


def outputs(m, x):
    h = m.maxpool(m.relu(m.bn1(m.conv1(x))))
    h = m.layer1(h)
    h = m.layer2(h)
    h = m.layer3(h)
    h = m.layer4(h)
    f = torch.flatten(m.avgpool(h), 1)
    return f, m.fc(f)


# DAN-DG averages marginal MMD over source-domain pairs; Sketch is not an input.
def mmd(a, b):
    joined = torch.cat([a, b])
    d = torch.cdist(joined, joined).square()
    med = d[d > 0].median().detach().clamp_min(1e-6)
    k = sum(torch.exp(-d / (2 * scale * med)) for scale in (0.5, 1, 2))
    n = len(a)
    return k[:n, :n].mean() + k[n:, n:].mean() - 2 * k[:n, n:].mean()


def loaders(split, tf):
    return {
        d: DataLoader(
            PACS(source[(source.domain == d) & (source.split == split)], tf),
            batch_size=CONFIG["batch_per_domain"],
            shuffle=True,
            drop_last=True,
        )
        for d in SOURCES
    }


def evaluate(m, frame):
    m.eval()
    freeze_bn(m)
    ys = []
    ps = []
    for x, y, _ in DataLoader(PACS(frame, eval_tf), batch_size=64):
        with torch.no_grad():
            _, z = outputs(m, x.to(DEVICE))
            ps.extend(z.argmax(1).cpu())
            ys.extend(y)
    return {
        "accuracy": accuracy_score(ys, ps),
        "macro_f1": f1_score(ys, ps, average="macro"),
        "y": np.array(ys),
        "pred": np.array(ps),
    }


## 3. ERM checkpoint reuse, DAN-DG, and SAM training

The next cell loads the Task 2 ERM checkpoint and trains DAN-DG and SAM. It records source-side classification and MMD curves, selecting checkpoints only from source-validation macro-F1.

In [ ]:
erm = make_model()
erm.load_state_dict(torch.load(CONFIG["erm_checkpoint"], map_location=DEVICE)["model"])


# SAM requires two passes: find a local loss-increasing perturbation, then update at that point.
def train(method, rho=CONFIG["rho"]):
    m = make_model()
    opt = torch.optim.AdamW(m.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    its = {d: iter(l) for d, l in loaders("train", train_tf).items()}
    best = -1
    stale = 0
    state = None
    history = []
    for epoch in range(CONFIG["epochs"]):
        m.train()
        freeze_bn(m)
        total_cls = total_mmd = 0
        steps = min(map(len, loaders("train", train_tf).values()))
        for _ in range(steps):
            batches = []
            for d in SOURCES:
                try:
                    b = next(its[d])
                except StopIteration:
                    its[d] = iter(loaders("train", train_tf)[d])
                    b = next(its[d])
                batches.append(b)
            x = torch.cat([b[0] for b in batches]).to(DEVICE)
            y = torch.cat([b[1] for b in batches]).to(DEVICE)
            f, z = outputs(m, x)
            cls = F.cross_entropy(z, y)
            penalty = (
                sum(
                    mmd(f[i * 8 : (i + 1) * 8], f[j * 8 : (j + 1) * 8])
                    for i, j in [(0, 1), (0, 2), (1, 2)]
                )
                / 3
                if method == "dan_dg"
                else torch.zeros((), device=DEVICE)
            )
            loss = cls + CONFIG["lambda_dg"] * penalty
            if method == "sam":
                opt.zero_grad()
                loss.backward()
                norm = torch.sqrt(
                    sum((p.grad**2).sum() for p in m.parameters() if p.grad is not None)
                )
                perturb = []
                with torch.no_grad():
                    for p in m.parameters():
                        e = (
                            rho * p.grad / (norm + 1e-12)
                            if p.grad is not None
                            else torch.zeros_like(p)
                        )
                        p.add_(e)
                        perturb.append(e)
                freeze_bn(m)
                _, z2 = outputs(m, x)
                opt.zero_grad()
                F.cross_entropy(z2, y).backward()
                with torch.no_grad():
                    for p, e in zip(m.parameters(), perturb):
                        p.sub_(e)
                opt.step()
            else:
                opt.zero_grad()
                loss.backward()
                opt.step()
            total_cls += cls.item()
            total_mmd += penalty.item()
        score = np.mean(
            [
                evaluate(m, source[(source.domain == d) & (source.split == "val")])["macro_f1"]
                for d in SOURCES
            ]
        )
        history.append(
            {
                "epoch": epoch + 1,
                "classification_loss": total_cls / steps,
                "mmd_penalty": total_mmd / steps,
                "mean_source_f1": score,
            }
        )
        if score > best:
            best, stale, state = score, 0, copy.deepcopy(m.state_dict())
        else:
            stale += 1
        if stale >= CONFIG["patience"]:
            break
    m.load_state_dict(state)
    return m, history


dan_dg, dan_history = train("dan_dg")
sam, sam_history = train("sam")


## 4. Source diagnostics only: metrics, source-domain separability, and sharpness proxy

The next cell computes source-domain separability and the common sharpness proxy using only source validation data, then saves the source diagnostic table.

In [ ]:
# Chance is 33.3% because this probe predicts one of three source domains.
def source_separability(m):
    frame = pd.concat(
        [
            source[(source.domain == d) & (source.split == "val")].assign(domain_id=i)
            for i, d in enumerate(SOURCES)
        ]
    )
    fs = []
    ds = []
    for x, _, d in DataLoader(PACS(frame, eval_tf), batch_size=64):
        with torch.no_grad():
            fs.append(outputs(m, x.to(DEVICE))[0].cpu())
            ds += [SOURCES.index(v) for v in d]
    ids = np.random.default_rng(SEED).permutation(len(ds))
    cut = int(0.7 * len(ids))
    clf = LogisticRegression(C=1, max_iter=2000, multi_class="auto").fit(
        torch.cat(fs)[ids[:cut]], np.array(ds)[ids[:cut]]
    )
    return clf.score(torch.cat(fs)[ids[cut:]], np.array(ds)[ids[cut:]])


def sharpness(m):
    fixed = pd.concat(
        [
            source[(source.domain == d) & (source.split == "val")].sample(32, random_state=SEED)
            for d in SOURCES
        ]
    )
    x, y, _ = next(iter(DataLoader(PACS(fixed, eval_tf), batch_size=96)))
    m.eval()
    freeze_bn(m)
    _, z = outputs(m, x.to(DEVICE))
    base = F.cross_entropy(z, y.to(DEVICE))
    m.zero_grad()
    base.backward()
    norm = torch.sqrt(sum((p.grad**2).sum() for p in m.parameters() if p.grad is not None))
    es = []
    with torch.no_grad():
        for p in m.parameters():
            e = 0.05 * p.grad / (norm + 1e-12) if p.grad is not None else torch.zeros_like(p)
            p.add_(e)
            es.append(e)
    with torch.no_grad():
        _, z = outputs(m, x.to(DEVICE))
        perturbed = F.cross_entropy(z, y.to(DEVICE))
    with torch.no_grad():
        for p, e in zip(m.parameters(), es):
            p.sub_(e)
    return (perturbed - base).item()


source_rows = []
for name, m in {"erm": erm, "dan_dg": dan_dg, "sam": sam}.items():
    scores = {
        d: evaluate(m, source[(source.domain == d) & (source.split == "val")]) for d in SOURCES
    }
    source_rows.append(
        {
            "method": name,
            **{f"{d}_accuracy": scores[d]["accuracy"] for d in SOURCES},
            "mean_source_accuracy": np.mean([s["accuracy"] for s in scores.values()]),
            "worst_source_accuracy": min(s["accuracy"] for s in scores.values()),
            "mean_source_macro_f1": np.mean([s["macro_f1"] for s in scores.values()]),
            "source_domain_separability": source_separability(m),
            "sharpness_proxy": sharpness(m),
        }
    )
pd.DataFrame(source_rows).to_csv(RESULTS / "source_diagnostics.csv", index=False)


## 5. Final Sketch evaluation (run only after all Task 3 configurations are fixed)

The next cell is the first permitted Sketch access. It produces final Sketch accuracy/macro-F1, relative changes, and the preconfigured SAM-strength study table.

In [ ]:
# This placement enforces the assignment's unseen-target protocol in the notebook flow.
# This is the first Task 3 code that accesses Sketch. Do not move it above training or source diagnostics.
sketch_records = []
for y, c in enumerate(CLASSES):
    sketch_records += [
        {"path": str(p), "domain": "sketch", "label": y}
        for p in sorted((CONFIG["pacs_root"] / "sketch" / c).glob("*"))
    ]
sketch = pd.DataFrame(sketch_records)
final = []
for row in source_rows:
    m = {"erm": erm, "dan_dg": dan_dg, "sam": sam}[row["method"]]
    result = evaluate(m, sketch)
    final.append(
        {**row, "sketch_accuracy": result["accuracy"], "sketch_macro_f1": result["macro_f1"]}
    )
final = pd.DataFrame(final)
final["sketch_accuracy_change_vs_erm"] = (
    final.sketch_accuracy - final.loc[final.method == "erm", "sketch_accuracy"].iloc[0]
)
final.to_csv(RESULTS / "final_metrics.csv", index=False)
if CONFIG["print_metrics"]:
    display(final)
# Controlled SAM study; settings are fixed before viewing this final-analysis metric.
study = []
for rho in CONFIG["study_rhos"]:
    candidate, _ = train("sam", rho)
    study.append(
        {
            "rho": rho,
            "mean_source_f1": np.mean(
                [
                    evaluate(candidate, source[(source.domain == d) & (source.split == "val")])[
                        "macro_f1"
                    ]
                    for d in SOURCES
                ]
            ),
            "sharpness_proxy": sharpness(candidate),
            "sketch_accuracy_final_analysis_only": evaluate(candidate, sketch)["accuracy"],
        }
    )
pd.DataFrame(study).to_csv(RESULTS / "sam_strength_study.csv", index=False)
